# Part 2 — Unbinned likelihood analysis

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from scipy.integrate import trapezoid
from scipy.optimize import root_scalar
from scipy.stats import norm, truncnorm, uniform

# Settings: font size for plot labels
matplotlib.rcParams['font.size'] = 14

Example in 1D. We consider values of an observable $x$ in the $[-10, 10]$ interval with a uniform background distribution and a Gaussian signal distribution.

In [ ]:
# PDF shapes

xmin, xmax = -10, 10
sigma = 1

# We use rv_continuous classes from scipy.stats, as they provide both the PDF computation (`.pdf(...)`) and the random generation (`.rvs(...)`)
sample_background_pdf = uniform(loc=xmin, scale=xmax-xmin)
sample_signal_pdf = truncnorm(xmin/sigma, xmax/sigma, loc=0, scale=sigma)

# Instead of these functions, you could also design your own classes defining the same methods
class example_pdf:
    def __init__(self, mu, sigma):
        self.mu, self.sigma = mu, sigma

    def pdf(self, x):
        return norm.pdf(x, loc=self.mu, scale=self.sigma)

    def rvs(self, size):
        return norm.rvs(loc=self.mu, scale=self.sigma, size=size)

In [ ]:
def generate_component(mu, pdf, fluctuate=True):
    n = np.random.poisson(lam=mu) if fluctuate else int(mu)
    return pdf.rvs(size=n)

Generate a dataset with $\mu_s,\mu_b$ the **expected** numbers of signal and background events. The actual numbers of signal and background events have to be randomly sampled from Poisson distributions.

In [ ]:
def generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf):
    signal = generate_component(mu_signal, signal_pdf)
    background = generate_component(mu_background, background_pdf)
    return np.concatenate((signal, background)), len(signal), len(background)

In [ ]:
mu_signal = 10
mu_background = 20

# Generate random dataset
data, nsig_true, nbkg_true = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)

if len(data)==0:
    print("Zero events generated in this dataset. Try again...")
    exit(1)

plt.hist(data, bins=20, range=(-10, 10), histtype="step")
plt.xlabel("x")
plt.ylabel("Number of events")
plt.show()

Log-likelihood function: use an **extended maximum likelihood** method, allowing for Poisson fluctuations of the numbers of signal and background events.

$$\mathcal{L} = e^{-(\mu_s + \mu_b)} \prod_i \mu_s S(x_i) + \mu_b B(x_i)$$

where $S(x), B(x)$ are the signal and background PDFs and $\mu_s,\mu_b$ are the **expected** numbers of signal and background events (which may differ from the observed ones due to Poisson fluctuations). The $x_i$ are the values of the observables for the dataset considered.

In [ ]:
def loglikelihood(data: np.ndarray, mu_signal: np.ndarray, mu_background: float, signal_pdf, background_pdf):
    """Return the log-likelihood for the given dataset.
    
    Arguments:
        data: array of size N, where N is the number of events in the dataset.
        mu_signal: array of size M for the expected signal, where M is the number of points for which the likelihood should be evaluated.
        mu_background: expected background.
        signal_pdf: object defining the signal PDF.
        background_pdf: object defining the background PDF.

    Returns:
        np.ndarray: array of size M with the likelihood at each point mu_signal.
    """
    l1 = mu_signal[:, np.newaxis] * signal_pdf.pdf(data) + (mu_background * background_pdf.pdf(data))[np.newaxis, :]
    return np.log(l1).sum(axis=1) - (mu_signal + mu_background)

**QUESTION 1:** Why is there `np.newaxis` in above formula?

### Posterior computation and usage

In Bayesian statistics, we need to define a prior for $\mu_\text{sig}$: $\pi(\mu_\text{sig})$.

Once defined, the posterior distribution is defined as:

$$ \pi(\mu_\text{sig} \vert \text{Data}) = \dfrac{\mathcal{L}(\text{Data} \vert \mu_\text{sig}) \times \pi(\mu_\text{sig})}{\int \mathcal{L}(\text{Data} \vert \mu) \times \pi(\mu) \textrm{d}\mu}$$

The denominator only serves as normalization, so it can be evaluated separately to ensure that $\int \pi(\mu_\text{sig} \vert \text{Data}) \text{d}\mu_\text{sig} = 1$.

In [ ]:
def prior_flat(mu, mu_max=100):
    return np.where((mu >=0) & (mu < mu_max), 1/mu_max, 0)

def posterior(data, mu_signal, mu_background, signal_pdf, background_pdf, prior):
    return np.exp(loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf)) * prior(mu_signal)

In [ ]:
x = np.linspace(0, 40, 1001)
ypost = posterior(data, x, mu_background, sample_signal_pdf, sample_background_pdf, prior_flat)
ypost /= trapezoid(ypost, x)

plt.plot(x, ypost)
plt.xlabel(r"$\mu_\text{signal}$")
plt.ylabel(r"Posterior $\pi(\mu_\text{sig} \vert \text{Data})$")
plt.ylim(0, None)
plt.show()

**QUESTION 2:** What does the `trapezoid` function do? Why is it important to do this normalization?

#### Importance of the prior

**QUESTION 3:** Define a Gaussian prior of mean 10 and sigma 2 and perform the analysis with it. Compare it with flat and Jeffreys' priors.

In [ ]:
def prior_jeffreys_poisson(mu, mu_max=100):
    return np.where((mu > 0) & (mu < mu_max), 1/(2*np.sqrt(mu*mu_max)), 0)

def prior_gaussian(mu, mean=10, sigma=2):
    # Define Gaussian prior
    return ...

ypost_jeff = posterior(data, x, mu_background, sample_signal_pdf, sample_background_pdf, prior_jeffreys_poisson)
ypost_jeff /= trapezoid(ypost_jeff, x)

# Compute the posterior
# ...
ypost_gaus = ...

fig, ax = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"wspace": 0.3})
ax[0].plot(x, prior_flat(x), label="Flat prior")
ax[0].plot(x, prior_jeffreys_poisson(x), label="Jeffreys prior")

# Plot the Gaussian prior
# ...
#
ax[1].plot(x, ypost)
ax[1].plot(x, ypost_jeff)

# Plot the posterior for a Gaussian prior
# ...

ax[0].set_xlabel(r"$\mu_\text{signal}$")
ax[0].set_ylabel(r"Prior $\pi(\mu_\text{sig})$")
ax[1].set_xlabel(r"$\mu_\text{signal}$")
ax[1].set_ylabel(r"Posterior $\pi(\mu_\text{sig} \vert \text{Data})$")
ax[0].set_ylim(0, None)
ax[1].set_ylim(0, None)
ax[0].legend(loc="best")
plt.show()

#### Compute upper limit and credible intervals

Credible intervals at 100X% credible level are built by finding a set $[\mu_\text{min}, \mu_\text{max}]$ such that:

$$\int_{\mu_\text{min}}^{\mu_\text{max}} \pi(\mu_\text{sig} \vert \text{Data}) \text{d}\mu_\text{sig} = X$$

Upper limit is a special case, where $\mu_\text{min}$ is set to the left edge of the parameter space ($=0$ here).

A general construction of a unique credible interval is to use the highest posterior density (HPD) interval is defined such that:

$$\int_{\mu_\text{sig}\in\text{HPD}} \pi(\mu_\text{sig} \vert \text{Data}) \text{d}\mu_\text{sig} = X$$
$$\forall \mu \in [\mu_\text{min}, \mu_\text{max}], \forall \mu' \notin [\mu_\text{min}, \mu_\text{max}], \pi(\mu \vert \text{Data}) \geq \pi(\mu' \vert \text{Data})$$

The interval may be disjoint if the posterior distribution is multimodal, but the interval is still uniquely defined in this case. The interval can be computed by:
1. Scanning values of $y$ and compute $\mathcal{I}(y) = \int_{\pi(\mu_\text{sig} \vert \text{Data}) > y} \pi(\mu_\text{sig} \vert \text{Data}) \text{d}\mu_\text{sig}$.
2. Find $y_\text{thr}$ such that $\mathcal{I}(y_\text{thr}) = X$.
3. The HPD interval is the set of $\mu$ such that $\pi(\mu \vert \text{Data}) > y_\text{thr}$.

In [ ]:
# This is I(y_thr) defined above
def get_integral_above_post(x, y, y_thr):
    return trapezoid(y[y>y_thr], x[y>y_thr])

# root_scalar is used to define the root of a function, here solving I(y) = 0.9 
y_thr = root_scalar(lambda thr: get_integral_above_post(x, ypost, thr) - 0.9, bracket=(0, 100)).root
mumin, mumax = min(x[ypost>y_thr]), max(x[ypost>y_thr])
print("90% HPD credible interval =", mumin, mumax)

**QUESTION 4:** How would you write the function to define the 90% upper limit?

In [ ]:
muul = ...
print("90% upper limit =", muul)

In [ ]:
plt.fill_between(x[ypost>y_thr], ypost[ypost>y_thr], color="blue", alpha=0.2, label="HPD interval")
# add 90% upper limit on the plot
# ...
plt.plot(x, ypost)
plt.axhline(y_thr, color="red", ls="dashed")
plt.axvline(mumin, color="red", ls="dotted")
plt.axvline(mumax, color="red", ls="dotted")

plt.xlabel(r"$\mu_\text{signal}$")
plt.ylabel(r"Posterior $\pi(\mu_\text{sig} \vert \text{Data})$")
plt.ylim(0, None)
plt.legend(loc="best")
plt.show()

#### Sensitivity computation

In Bayesian analysis, the concepts of sensitivity and discovery potential do not necessarily make sense. However, we may want to be able to estimate the expected upper limit, assuming data follow the background-only hypothesis. Then, we perform background-only pseudo-experiments, compute the upper limit each time, and report the median.

In [ ]:
def run_one_pseudo_experiment(mu_signal, mu_background, signal_pdf, background_pdf, prior, mu_signal_max=40):
    data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
    x = np.linspace(0, mu_signal_max, 1001)
    ypost = posterior(data, x, mu_background, signal_pdf, background_pdf, prior)
    ypost /= trapezoid(ypost, x)
    return x, ypost

trace_posteriors = []
uls = []
for i in range(1000):
    x, ypost = run_one_pseudo_experiment(0, mu_background, sample_signal_pdf, sample_background_pdf, prior_flat)
    uls.append(root_scalar(lambda mu: trapezoid(ypost[x<mu], x[x<mu]) - 0.9, bracket=(0, 100)).root)
    if i < 10:
        trace_posteriors.append((x, ypost))

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for post in trace_posteriors:
    ax[0].plot(*post)
    ax[0].set_xlabel(r"$\mu_\text{signal}$")
    ax[0].set_ylabel(r"Posterior $\pi(\mu_\text{sig} \vert \text{Data})$")
ax[0].set_xlim(0, 20)
ax[0].set_ylim(0, None)
ax[0].set_title("Example posteriors")
ax[1].hist(uls, bins=20, histtype="step")
ax[1].axvline(np.median(uls), color="red", label="Median")
ax[1].set_xlabel("90% upper limit")
ax[1].set_title("Upper limits")
ax[1].legend(loc="best")

print(f"Sensitivty = median 90% upper limit = {np.median(uls):.2f}")

**QUESTION 5**: What is the uncertainty on the median? How would you estimate it?

In [ ]:
# ...

### Bayes factor

The concept of "p-values" in frequentist approaches (quantifying rejection of a given hypothesis) is "replaced" by the computation of Bayes factors, aiming at updating our preference between competing models.

$$\dfrac{\pi(M_1 \vert D)}{\pi(M_2 \vert D)} = \dfrac{\int \mathcal{L}(D \vert \theta_1, M_1) \pi(\theta_1 \vert M_1) {\rm d}\theta_1}{\int \mathcal{L}(D \vert \theta_2, M_2) \pi(\theta_2 \vert M_2) {\rm d}\theta_2}	\times \dfrac{\pi(M_1)}{\pi(M_2)} = B_{12} \times \dfrac{\pi(M_1)}{\pi(M_2)}$$

The terms that appear in the Bayes factor are simply the integrals we were computing before to normalise the distribution. They are called evidence.

In our case, for the background-only hypothesis, there are no signal parameter and the evidence is simply $\mathcal{L}(D \vert \mu=0)$.

As done for p-values, we can define thresholds on Bayes factor corresponding to strong/decisive evidence for one model or the other. Jeffreys' scale is one such scale:
- $\text{BF} > 10$ or $\text{BF} < 0.1$ ($\vert\log_{10}\text{BF}\vert > 1$) → strong
- $\text{BF} > 100$ or $\text{BF} < 0.01$ ($\vert\log_{10}\text{BF}\vert > 2$) → decisive

In [ ]:
def compute_bayesfactor(data, mu_background, signal_pdf, background_pdf, prior, mu_signal_max=40):
    x = np.linspace(0, mu_signal_max, 1001)
    ypost = posterior(data, x, mu_background, signal_pdf, background_pdf, prior)
    evidence_H1 = trapezoid(ypost, x)
    evidence_H0 = np.exp(loglikelihood(data, np.zeros(1), mu_background, signal_pdf, background_pdf))[0]
    return evidence_H1 / evidence_H0

mus = np.arange(10+1)
npes = 500
bfs = np.zeros((len(mus), npes))
for i, mu_signal in enumerate(mus):
    for j in range(npes):
        data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
        bfs[i, j] = compute_bayesfactor(data, mu_background, sample_signal_pdf, sample_background_pdf, prior_flat)

plt.boxplot(np.log10(bfs).T, '.', positions=mus)
plt.axhline(1, color="red", ls="dotted")
plt.axhline(2, color="red", ls="dashed")
plt.xlabel("Injected signal")
plt.ylabel(r"$\log_{10}$(Bayes factor)")

An important note is that the exact value of the Bayes factor strongly depends on the prior used, though the trend may be the same for different prior choices. If the data starts to be strongly constraining (e.g., in the presence of a large signal), the choice of (any reasonable) prior should not change the result.

In [ ]:
npes = 100
mu_signal = 5
bfs_flat, bfs_jeff = [], []
for _ in range(npes):
    data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
    bfs_flat.append(compute_bayesfactor(data, mu_background, sample_signal_pdf, sample_background_pdf, prior_flat))
    bfs_jeff.append(compute_bayesfactor(data, mu_background, sample_signal_pdf, sample_background_pdf, prior_jeffreys_poisson))

plt.scatter(np.log10(bfs_flat), np.log10(bfs_jeff))
line = [min(np.log10(bfs_flat+bfs_jeff)), max(np.log10(bfs_flat+bfs_jeff))]
plt.plot(line, line, color="black", ls="dotted")
plt.xlabel(r"$\log_{10}$(Bayes factor) flat")
plt.ylabel(r"$\log_{10}$(Bayes factor) Jeffreys")

**QUESTION 6:** We define a given dataset, compute the corresponding Bayes factor assuming a flat prior with `mu_max = 100`?

**QUESTION 7:** Recompute the Bayes factor for different `mu_max`. What do you observe?

In [ ]:
mu_background = 20
data0 = [-1.0199454386807156, -0.6143812005394883, 2.9912620947165216, -1.6013819345831592, 8.835618923532799, -0.36393538590560226, 0.0919319937840021, -1.8808342318076043, 0.04946987877027699, 9.553954092047718, -9.287369500095517, 4.599607770036229, -4.469144214560039, -2.0830749454382156, 3.3288522635862456, -1.9969803387585192, 0.8257765685685081, 1.2862965380657543, 8.199303818430856, 2.989396875772787, 0.9873283834154969, -4.519523834188595, -2.6518710585162015, 8.834758780563202]

# ...

### Inclusion of nuisance parameters

Nuisance parameters are simply added in the posterior definition:
- the likelihood is updated to account for the additional varying parameter
- a prior is added to incorporate any prior knowledge on the parameter

An example following the case presented above is the addition of uncertainties on the expected number of background events. The posterior distribution would then depend on both $\mu_\text{sig}$ and $\mu_\text{bkg}$. As we may be interested only in $\mu_\text{sig}$, we can decide to integrate on the other axis:

$$\pi(\mu_\text{sig} \vert \text{Data}) = \int \pi(\mu_\text{sig}, \mu_\text{bkg} \vert \text{Data}) \text{d}\mu_\text{bkg}$$

From there, credible intervals, sensitivities... can be derived as before.

In [ ]:
def sample_prior_bkg(mu_background, mean=20, sigma=5):
    return truncnorm.pdf(mu_background, -mean/sigma, np.inf, mean, sigma)

def posterior_with_nuisance(data, mu_signal, mu_background, signal_pdf, background_pdf, prior_sig, prior_bkg):
    return np.exp(loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf)) * prior_sig(mu_signal) * prior_bkg(mu_background)

data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
x = np.linspace(0, 40, 1001)
y = np.linspace(0, 40, 1001)
zpost = []
for _y in y:
    zpost.append(posterior_with_nuisance(data, x, _y, sample_signal_pdf, sample_background_pdf, prior_flat, prior_bkg=sample_prior_bkg))

# Integrate over mubkg (here we can simply sum over the axis of the bkg, as the sampling points are equidistant and normalization will be done after)
zpost_1d = np.sum(zpost, axis=0)
zpost_1d /= trapezoid(zpost_1d, x)

fig, ax = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
ax[0].pcolormesh(x, y, zpost, shading="nearest", cmap="Blues")
ax[0].set_ylabel(r"$\mu_\text{bkg}$")
ax[1].plot(x, zpost_1d)
ax[1].set_xlabel(r"$\mu_\text{sig}$")
ax[1].set_ylabel(r"Posterior $\pi(\mu_\text{sig} \vert \text{Data})$")
plt.show()

**QUESTION 8:** You may also be interested in the posterior $\pi(\mu_\text{bkg} \vert \text{Data})$. How can you compute it?

**QUESTION 9:** How would you draw a confidence contour on the 2D plot? Which method would guarantee a unique contour to be defined?